

Revised by M.D.E. for "87450 - MODELS AND NUMERICAL METHODS IN PHYSICS"


# Boids Flockers

An implementation of Craig Reynolds's Boids flocker model. Agents (simulated birds) try to fly towards the average position of their neighbors and in the same direction as them, while maintaining a minimum distance. This produces flocking behavior.

This model tests Mesa's continuous space feature, and uses numpy arrays to represent vectors.

### Set Up

In [ ]:
!pip install --quiet mesa[rec]

### Import Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import mesa

## Agent

Each Boid agent follows three simple local rules:
- **Cohesion**: steer toward the average position of neighbours
- **Separation**: avoid getting too close to any neighbour
- **Alignment**: steer toward the average heading of neighbours

In [ ]:
from mesa.experimental.continuous_space import ContinuousSpaceAgent

## Where does `ContinuousSpaceAgent` come from?
```python
from mesa.experimental.continuous_space import ContinuousSpaceAgent
```

This single line tells Python to load a specialised base class from Mesa's library.
To read it, parse it right-to-left:

| Fragment | Meaning |
|---|---|
| `mesa` | the Mesa package (installed with `pip install mesa`) |
| `.experimental` | a sub-package of Mesa containing features still under active development — the API may change between versions |
| `.continuous_space` | a module inside `experimental` dedicated to agents that live in a **continuous** (real-valued) space, not a discrete grid |
| `ContinuousSpaceAgent` | the specific class we import — the blueprint for any agent that has a position vector in ℝ² |

### Why a specialised base class?

In the Boltzmann Wealth Model every agent was just a `mesa.Agent` — it had no notion of position at all.
Here each boid lives at a point **x** = (x, y) ∈ [0, L]², moves continuously, and needs to *query its spatial neighbourhood*.
`ContinuousSpaceAgent` adds exactly that machinery on top of the standard `mesa.Agent`:

- it stores `self.position` as a NumPy array
- it exposes `self.get_neighbors_in_radius(radius)` — a spatial query that returns all agents within distance `r`
- it handles the **torus boundary** (periodic boundary conditions) transparently

### The `experimental` label

This does **not** mean the code is broken or untested.
It means the Mesa developers have not yet frozen the interface — a future Mesa release could rename a method or change a default.
For our purposes it works perfectly; just pin your Mesa version in the notebook header (`pip install mesa==3.x`).

### Physics analogy

| Mesa concept | Physics analogue |
|---|---|
| `mesa.Agent` | a structureless particle (no position) |
| `ContinuousSpaceAgent` | a particle in phase space with coordinates **x** |
| `get_neighbors_in_radius(r)` | interaction cutoff — only particles within range `r` feel the force |
| torus boundary | periodic boundary conditions (standard in MD / Vicsek model) |

In [ ]:



class Boid(ContinuousSpaceAgent):
    """A Boid-style flocker agent."""

    def __init__(
        self,
        model,
        space,
        position=(0, 0),
        speed=1,
        direction=(1, 1),
        vision=1,
        separation=1,
        cohere=0.03,
        separate=0.015,
        match=0.05,
    ):
        super().__init__(space, model)
        self.position = np.array(position, dtype=float)
        self.speed = speed
        self.direction = np.array(direction, dtype=float)
        self.vision = vision
        self.separation = separation
        self.cohere_factor = cohere
        self.separate_factor = separate
        self.match_factor = match
        self.neighbors = []
        self.angle = 0.0

    def step(self):
        """Get the Boid's neighbors, compute the new vector, and move accordingly."""
        neighbors, distances = self.get_neighbors_in_radius(radius=self.vision)
        self.neighbors = [n for n in neighbors if n is not self]

        if not self.neighbors:
            self.position += self.direction * self.speed
            return

        delta = self.space.calculate_difference_vector(self.position, agents=self.neighbors)
        distances_nb = distances[1:] if len(distances) > 1 else distances  # skip self

        cohere_vector = delta.sum(axis=0) * self.cohere_factor
        norms = np.linalg.norm(delta, axis=1)
        separation_vector = -1 * delta[norms < self.separation].sum(axis=0) * self.separate_factor
        match_vector = (
            np.asarray([n.direction for n in self.neighbors]).sum(axis=0) * self.match_factor
        )

        self.direction += (cohere_vector + separation_vector + match_vector) / len(self.neighbors)

        norm = np.linalg.norm(self.direction)
        if norm > 0:
            self.direction /= norm

        self.position += self.direction * self.speed
        self.angle = np.degrees(np.arctan2(self.direction[1], self.direction[0]))



## The `Boid.__init__` — initial conditions and the `super()` call


---

### Reading the signature

The first line after `def __init__(self, ...` is the **parameter list** — everything Mesa (or you) can pass when creating a boid.
```
self          → the boid being created  ("particle index i")
model         → the BoidFlockers simulation box
space         → the ContinuousSpace arena the boid lives in
position      → starting coordinates, default (0,0)
speed         → step size per tick, default 1
direction     → unit vector of initial heading, default (1,1)
vision        → interaction radius r, default 1
separation    → minimum distance before repulsion kicks in
cohere        → strength of cohesion rule
separate      → strength of separation rule
match         → strength of alignment rule
```

Parameters with `= value` are **optional** — if the caller does not supply them, the default is used.
Parameters without a default (`self`, `model`, `space`) are **mandatory**.

---

### The `super().__init__(space, model)` line

This is the most important line in the whole `__init__`.

`super()` means *"the parent class"* — here that is `ContinuousSpaceAgent`.
Calling `super().__init__(space, model)` runs **the parent's own setup** before we add anything of our own.

What does the parent do with those two arguments?

| Argument passed | What `ContinuousSpaceAgent.__init__` does with it |
|---|---|
| `space` | registers this boid in the spatial index so `get_neighbors_in_radius` works |
| `model` | calls `mesa.Agent.__init__(model)` further up the chain, which assigns `self.unique_id` and adds the agent to `model.agents` |

**Rule**: `super().__init__(...)` must always be the first line of your `__init__`.
If you set `self.position` before it, the parent constructor may overwrite it.
If you skip it entirely, the boid is never registered in the space — `get_neighbors_in_radius` will silently return nothing.

The chain of calls looks like this:
```
Boid.__init__()
  └─▶  ContinuousSpaceAgent.__init__(space, model)
         └─▶  mesa.Agent.__init__(model)
                └─▶  assigns self.unique_id
                └─▶  adds self to model.agents
```

Physics analogy: `super().__init__()` is like calling the base equations of motion before adding your specific interaction potential.
The base class handles bookkeeping (particle index, neighbour list registration);
you handle physics (mass, velocity, force constants).

---

### The state variables

After `super()`, every line sets **one attribute of this specific boid** — its initial conditions at t = 0.
```python
self.position  = np.array(position,  dtype=float)  # x⃗(0)
self.direction = np.array(direction, dtype=float)  # v̂(0)  — unit vector, normalised later
self.speed     = speed                             # |v| — fixed scalar
```

`position` and `direction` are stored as **NumPy arrays**, not Python tuples.
This matters: the `step()` method will do vector arithmetic (`self.position += self.direction * self.speed`).
NumPy arrays support that; plain tuples do not.

The three *factor* attributes are the coupling constants of the three boid rules:
```python
self.cohere_factor   = cohere    # κ_c  — cohesion strength
self.separate_factor = separate  # κ_s  — repulsion strength
self.match_factor    = match     # κ_m  — alignment strength
```

They are set once here and never change during the simulation — they are **quenched parameters**, like a fixed interaction potential.

Finally:
```python
self.neighbors = []    # will be populated at each step()
self.angle     = 0.0   # heading in degrees — used only for visualisation
```

`neighbors` starts empty and is recomputed from scratch at every time step.
`angle` is a derived quantity (not used in the dynamics) kept for convenience when drawing the arrows.

## `Boid.step()` — the three rules of flocking

This is the entire physics of the model. It runs once per boid per time step.

---

### Phase 1 — find the neighbours
```python
neighbors, distances = self.get_neighbors_in_radius(radius=self.vision)
self.neighbors = [n for n in neighbors if n is not self]
```

`get_neighbors_in_radius` queries the spatial index of the `ContinuousSpace`
and returns every agent within Euclidean distance `self.vision` — including the
boid itself (it is inside its own radius by definition).

The list comprehension `if n is not self` removes the boid from its own
neighbour list.  `is not self` tests **object identity**, not equality — it
asks "is this the exact same Python object?", which is exactly what we want.
```python
if not self.neighbors:
    self.position += self.direction * self.speed
    return
```

If the boid is isolated it simply keeps flying in its current direction and
exits `step()` early (`return` skips everything below).
This is the **free-streaming** case — no interaction, no direction change.

---

### Phase 2 — compute the difference vectors
```python
delta = self.space.calculate_difference_vector(self.position, agents=self.neighbors)
```

`delta` is a 2-D NumPy array of shape `(n_neighbours, 2)`.
Row `k` is the vector from **this boid** to neighbour `k`:
```
delta[k] = neighbour_k.position − self.position
```

The method is called on `self.space` (not on `self`) because the space knows
about the **torus topology** — if a neighbour is on the other side of the
periodic boundary, the naive difference would be wrong.
`calculate_difference_vector` returns the **minimum-image** vector, exactly as
in molecular dynamics with periodic boundary conditions.

---

### Phase 3 — three forces, one update

The new direction is the sum of three vectorial contributions.
Think of them as three forces acting on the heading, not on the position.

#### Rule 1 — Cohesion  (fly toward the centre of mass)
```python
cohere_vector = delta.sum(axis=0) * self.cohere_factor
```

`delta.sum(axis=0)` sums all displacement vectors → result is a single 2-D
vector pointing from this boid toward the **centroid** of its neighbours.

Multiplying by `self.cohere_factor` (κ_c = 0.03) scales the strength.

| symbol | code |
|--------|------|
| **c** = κ_c Σ_j (**x**_j − **x**_i) | `cohere_vector` |

#### Rule 2 — Separation  (avoid collisions)
```python
norms = np.linalg.norm(delta, axis=1)
separation_vector = -1 * delta[norms < self.separation].sum(axis=0) * self.separate_factor
```

`np.linalg.norm(delta, axis=1)` computes the scalar distance to each neighbour
— one number per row, result shape `(n_neighbours,)`.

`delta[norms < self.separation]` is **boolean indexing**: it selects only the
rows where the neighbour is *too close* (inside the separation radius).

The minus sign reverses those vectors → the boid is pushed **away** from
anyone inside the danger zone.

| symbol | code |
|--------|------|
| **s** = −κ_s Σ_{j: \|Δ\|<d_sep} (**x**_j − **x**_i) | `separation_vector` |

#### Rule 3 — Alignment  (match the flock's heading)
```python
match_vector = np.asarray([n.direction for n in self.neighbors]).sum(axis=0) * self.match_factor
```

`[n.direction for n in self.neighbors]` collects the direction vector of every
neighbour into a Python list, then `np.asarray(...)` turns it into a
`(n_neighbours, 2)` array.  `.sum(axis=0)` gives the **total heading** of the
neighbourhood.

| symbol | code |
|--------|------|
| **m** = κ_m Σ_j **v̂**_j | `match_vector` |

---

### Phase 4 — update direction and normalise
```python
self.direction += (cohere_vector + separation_vector + match_vector) / len(self.neighbors)
```

The three vectors are added and divided by the number of neighbours
(averaging, not summing) to make the update **independent of flock density**.
Without the division a boid surrounded by 50 neighbours would be pushed 50×
harder than one with 1 neighbour.
```python
norm = np.linalg.norm(self.direction)
if norm > 0:
    self.direction /= norm
```

**Normalisation**: after adding the correction the direction vector is no
longer a unit vector.  Dividing by its norm restores `|self.direction| = 1`.

This is the Vicsek constraint: every boid flies at the **same fixed speed**,
only the heading changes.  Speed is not a dynamical variable.

---

### Phase 5 — move and update angle
```python
self.position += self.direction * self.speed
self.angle = np.degrees(np.arctan2(self.direction[1], self.direction[0]))
```

`self.direction` is now a unit vector, so multiplying by `self.speed` gives a
displacement of exactly `speed` units per step — **Euler integration** with
step size `speed`.

`self.angle` converts the heading to degrees for the visualisation arrows.
It plays no role in the dynamics.

---

### The full update equation

Putting it together, at each step the heading evolves as:
```
v̂ᵢ(t+1)  ∝  v̂ᵢ(t)
            + κ_c  Σⱼ (xⱼ − xᵢ)          ← cohesion
            − κ_s  Σ_{j: |Δ|<d} (xⱼ − xᵢ) ← separation
            + κ_m  Σⱼ v̂ⱼ                  ← alignment
```

then normalised to unit length, then:
```
xᵢ(t+1) = xᵢ(t) + speed · v̂ᵢ(t+1)
```

This is a **soft** version of the original Vicsek model: instead of
snapping to the exact mean heading plus noise, the direction is nudged
continuously by three competing terms.  The Vicsek order parameter φ = |⟨v̂ᵢ⟩|
measures how well all those nudges have aligned the flock globally.

## Model

In [ ]:
from mesa import Model
from mesa.experimental.continuous_space import ContinuousSpace


class BoidFlockers(Model):
    """Flocker model class. Handles agent creation, placement and scheduling."""

    def __init__(
        self,
        population_size=100,
        width=100,
        height=100,
        speed=1,
        vision=10,
        separation=2,
        cohere=0.03,
        separate=0.015,
        match=0.05,
        seed=None,
    ):
        super().__init__(seed=seed)

        self.space = ContinuousSpace(
            [[0, width], [0, height]],
            torus=True,
            random=self.random,
            n_agents=population_size,
        )

        positions = self.rng.random(size=(population_size, 2)) * self.space.size
        directions = self.rng.uniform(-1, 1, size=(population_size, 2))

        Boid.create_agents(
            self,
            population_size,
            self.space,
            position=positions,
            direction=directions,
            cohere=cohere,
            separate=separate,
            match=match,
            speed=speed,
            vision=vision,
            separation=separation,
        )

        self.average_heading = None
        self.update_average_heading()

    def update_average_heading(self):
        if not self.agents:
            self.average_heading = 0
            return
        headings = np.array([agent.direction for agent in self.agents])
        mean_heading = np.mean(headings, axis=0)
        self.average_heading = np.arctan2(mean_heading[1], mean_heading[0])

    def step(self):
        self.agents.shuffle_do("step")
        self.update_average_heading()

## `BoidFlockers.__init__` — building the simulation box

---

### Imports
```python
from mesa import Model
from mesa.experimental.continuous_space import ContinuousSpace
```

Two things are imported:

| Name | Role |
|------|------|
| `Model` | Mesa's base class for the simulation — manages `model.agents`, the RNG, the step counter |
| `ContinuousSpace` | the spatial arena: a 2-D real-valued box with a built-in neighbour index |

---

### The signature
```python
class BoidFlockers(Model):
    def __init__(
        self,
        population_size=100,
        width=100, height=100,
        speed=1, vision=10, separation=2,
        cohere=0.03, separate=0.015, match=0.05,
        seed=None,
    ):
```

All parameters have defaults — you can instantiate the model with just
`BoidFlockers()` and get a reasonable simulation.
`seed` controls the random number generator; passing a fixed integer gives a
**reproducible** run (same initial positions every time).

---

### Step 1 — initialise the base class
```python
super().__init__(seed=seed)
```

Same pattern as in `Boid.__init__`.
Calling `Model.__init__` sets up:

- `self.agents` — the AgentSet that will hold all boids
- `self.random` — a seeded Python `random.Random` instance
- `self.rng` — a seeded `numpy.random.Generator` instance
- `self.steps` — step counter, starts at 0

The `seed` is forwarded here so both `self.random` and `self.rng` are
seeded consistently.  After this line we can safely use `self.rng`.

---

### Step 2 — create the arena
```python
self.space = ContinuousSpace(
    [[0, width], [0, height]],
    torus=True,
    random=self.random,
    n_agents=population_size,
)
```

`ContinuousSpace` builds a 2-D real-valued box.

| Argument | Meaning |
|----------|---------|
| `[[0, width], [0, height]]` | bounding box: x ∈ [0, 100], y ∈ [0, 100] |
| `torus=True` | periodic boundary conditions — an agent leaving the right edge re-enters from the left |
| `random=self.random` | passes the model's seeded RNG to the space (used internally for tie-breaking) |
| `n_agents=population_size` | pre-allocates the spatial index for N agents (performance hint) |

`torus=True` is the standard choice in Vicsek-type models: there are no walls,
no boundary effects, only bulk behaviour.

The space object is stored as `self.space` so both the model and the agents can
reach it.

---

### Step 3 — draw initial conditions
```python
positions  = self.rng.random(size=(population_size, 2)) * self.space.size
directions = self.rng.uniform(-1, 1, size=(population_size, 2))
```

Both are NumPy arrays of shape `(N, 2)` — one row per boid.

**`positions`**

`self.rng.random(size=(N, 2))` draws N×2 values uniformly in [0, 1).
Multiplying by `self.space.size` (a 2-element array `[width, height]`)
rescales to the actual box dimensions.
Result: N random points scattered uniformly in the arena.

**`directions`**

`self.rng.uniform(-1, 1, size=(N, 2))` draws components uniformly in [−1, 1).
These are **not** unit vectors yet — normalisation happens later inside
`Boid.step()` at the first time step.
Using uniform components in [−1, 1] gives random directions with no preferred
orientation (isotropic initial condition).

Both arrays are created here, **before** any agent exists, so they can be
passed as batches to `create_agents`.

---

### Step 4 — create all boids at once
```python
Boid.create_agents(
    self,                    # the model
    population_size,         # how many to create
    self.space,              # mandatory for ContinuousSpaceAgent
    position=positions,      # (N, 2) array — one row per boid
    direction=directions,    # (N, 2) array — one row per boid
    cohere=cohere,
    separate=separate,
    match=match,
    speed=speed,
    vision=vision,
    separation=separation,
)
```

`Boid.create_agents` is a **class method** provided by Mesa.
It calls `Boid.__init__` once per agent, distributing the rows of
`positions` and `directions` automatically — boid 0 gets `positions[0]`
and `directions[0]`, boid 1 gets `positions[1]` and `directions[1]`, and so on.

Scalar arguments (`cohere`, `speed`, …) are broadcast — every boid gets
the same value.

After this line `self.agents` contains `population_size` fully initialised
`Boid` objects, each registered in `self.space`.

**Why a class method and not a loop?**
You could write:
```python
for i in range(population_size):
    Boid(self, self.space, position=positions[i], ...)
```
`create_agents` does exactly that internally, but more efficiently and
without the boilerplate. It is the Mesa idiom.

---

### Step 5 — compute the initial order parameter
```python
self.average_heading = None
self.update_average_heading()
```

`average_heading` is the model-level observable — the mean heading angle of
the whole flock.  It is initialised to `None` and then immediately computed
via `update_average_heading()` so the value is valid before any step is taken.

---

### `update_average_heading`
```python
def update_average_heading(self):
    headings = np.array([agent.direction for agent in self.agents])
    mean_heading = np.mean(headings, axis=0)
    self.average_heading = np.arctan2(mean_heading[1], mean_heading[0])
```

This computes the **mean velocity vector** of the flock and converts it to
an angle in radians.
```
mean_heading = (1/N) Σᵢ v̂ᵢ      ← a 2-D vector
average_heading = arctan2(y, x)   ← angle of that vector, in [−π, π]
```

Note: `average_heading` is the **angle** of the mean vector, not the mean
of the angles — these are different things.
Taking the mean vector first and then the angle is the correct circular
statistic (it handles the wrap-around at ±π properly).

The Vicsek order parameter φ = |mean_heading| (its magnitude) is not stored
here but is trivial to compute:
```python
phi = np.linalg.norm(np.mean(headings, axis=0))
```

---

### `BoidFlockers.step`
```python
def step(self):
    self.agents.shuffle_do("step")
    self.update_average_heading()
```

Two lines, two jobs:

| Line | What it does |
|------|-------------|
| `self.agents.shuffle_do("step")` | randomly reorders the agent list, then calls `boid.step()` on each agent in that order |
| `self.update_average_heading()` | recomputes the order parameter after all agents have moved |

`shuffle_do` implements **random sequential update**: at each time step the
agents do not all move simultaneously — they move one at a time in a random
order.  This avoids artefacts from a fixed activation sequence and is the
standard update rule in Vicsek-type ABMs.

---

### Complete data-flow summary
```
BoidFlockers.__init__()
│
├─ super().__init__(seed)       → self.agents, self.rng, self.random
├─ ContinuousSpace(...)         → self.space  (torus arena + spatial index)
├─ self.rng.random(...)         → positions   (N×2 array)
├─ self.rng.uniform(...)        → directions  (N×2 array)
├─ Boid.create_agents(...)      → N Boid objects, each in self.space
└─ update_average_heading()     → self.average_heading (initial value)

BoidFlockers.step()
├─ shuffle_do("step")           → each Boid.step() runs once, random order
└─ update_average_heading()     → self.average_heading updated
```

## Visualisation — Matplotlib Animation

> **Note:** Mesa's built-in [`SolaraViz`](https://mesa.readthedocs.io/latest/apis/visualization.html) requires a running web server and does **not** work in Colab.
> We replace it with a standard `matplotlib` animation — fully self-contained and Colab-friendly.

In [ ]:
N_BOIDS = 80; WIDTH = 100; HEIGHT = 100; SPEED = 2; VISION = 10; SEPARATION = 2; N_STEPS = 150

model = BoidFlockers(
    population_size=N_BOIDS, width=WIDTH, height=HEIGHT,
    speed=SPEED, vision=VISION, separation=SEPARATION, seed=42,
)

fig, ax = plt.subplots(figsize=(6, 6))
fig.patch.set_facecolor("#111111")

def update(frame):
    if frame > 0:
        model.step()
    ax.clear()
    ax.set_xlim(0, WIDTH); ax.set_ylim(0, HEIGHT)
    ax.set_facecolor("#111111"); ax.set_xticks([]); ax.set_yticks([])

    agents   = list(model.agents)
    pos      = np.array([a.position  for a in agents])
    flocking = np.array([len(a.neighbors) >= 2 for a in agents])

    for mask, color in [(~flocking, "#E05252"), (flocking, "#52C07A")]:
        pts = pos[mask]
        if len(pts):
            ax.scatter(pts[:, 0], pts[:, 1], s=20, c=color, zorder=2)

    ax.set_title(f"Step {frame:3d}  |  {100*flocking.mean():.0f}% flocking",
                 color="white", fontsize=11)

ani = animation.FuncAnimation(fig, update, frames=N_STEPS, interval=80)
plt.tight_layout()
HTML(ani.to_jshtml())

## Order Parameter — Measuring Collective Motion

In physics, the **Vicsek order parameter** $\phi = |\langle \hat{v}_i \rangle|$ measures how aligned the flock is:
- $\phi \approx 0$ → disordered (random directions)
- $\phi \approx 1$ → fully aligned (perfect flock)

This is directly analogous to the **magnetisation** in the Ising model.

In [ ]:
# Run a fresh simulation and track the order parameter over time
model2 = BoidFlockers(
    population_size=150, width=100, height=100,
    speed=2, vision=10, separation=2, seed=0
)

order_param = []
for _ in range(200):
    model2.step()
    dirs = np.array([a.direction for a in model2.agents])
    mean_v = np.mean(dirs, axis=0)
    order_param.append(np.linalg.norm(mean_v))

fig2, ax2 = plt.subplots(figsize=(8, 3))
ax2.plot(order_param, color="#4A9ECC", lw=2)
ax2.axhline(1.0, color="white", lw=0.8, ls="--", alpha=0.4, label="perfect order")
ax2.set_xlabel("Step", color="white")
ax2.set_ylabel("Order parameter φ", color="white")
ax2.set_title("Collective alignment (Vicsek order parameter)", color="white")
ax2.set_facecolor("#1A1A1A")
fig2.patch.set_facecolor("#1A1A1A")
ax2.tick_params(colors="white")
ax2.spines[:].set_color("#444444")
ax2.legend(facecolor="#2D2D2D", labelcolor="white")
plt.tight_layout()
plt.show()

## Key references on the Boids / Vicsek family of models

---

### Reynolds, C. W. (1987)
*Flocks, herds and schools: A distributed behavioral model.*
SIGGRAPH Computer Graphics, 21(4), 25–34.

The paper that introduces the three rules — cohesion, separation, alignment —
as an algorithm for computer graphics. There is no physics here, no order
parameter, no thermodynamic limit. Reynolds was trying to animate birds
convincingly on screen. The fact that three purely local rules produce
realistic collective motion was the conceptual surprise, and it is the
direct inspiration for the `Boid.step()` method in our notebook.

---

### Vicsek, T., Czirók, A., Ben-Jacob, E., Cohen, I., & Shochet, O. (1995)
*Novel type of phase transition in a system of self-propelled particles.*
Physical Review Letters, 75(6), 1226–1229.

**The** foundational reference for the physics of flocking.
The model is minimal: N point particles move at fixed speed v₀,
and at each step each particle sets its heading to the average heading
of its neighbours plus a noise term η.
The key result is that as noise decreases (or density increases) the system
undergoes a transition from a disordered phase (φ ≈ 0, random directions)
to an ordered phase (φ ≈ 1, collective motion).
The order parameter φ = |⟨v̂ᵢ⟩| is explicitly analogous to magnetisation
in the Ising model — this analogy is made in the paper itself.
One of the most cited papers in statistical physics of the last thirty years
(~6000 citations). Our notebook's order parameter cell measures exactly this φ.

---

### Toner, J., & Tu, Y. (1995)
*Long-range order in a two-dimensional dynamical XY model: How birds fly together.*
Physical Review Letters, 75(23), 4326–4329.

### Toner, J., & Tu, Y. (1998)
*Flocks, herds, and schools: A quantitative theory of flocking.*
Physical Review E, 58(4), 4828–4858.

These two papers derive the **continuum hydrodynamic equations** for the
ordered phase: the velocity field of the flock obeys equations analogous to
those of a polar active fluid.
The striking result is that long-range orientational order survives in 2D,
which appears to violate the Mermin-Wagner theorem.
The resolution is that the system is out of equilibrium — the theorem does
not apply. This is one of the earliest and clearest demonstrations that
active matter can sustain ordered phases forbidden in equilibrium systems
of the same dimensionality.

---

### Chaté, H., Ginelli, F., Grégoire, G., & Raynaud, F. (2008)
*Collective motion of self-propelled particles interacting without cohesion.*
Physical Review E, 77(4), 046113.

The original Vicsek paper suggested the disorder-to-order transition was
continuous (second order), analogous to a ferromagnetic transition.
This paper uses large-scale numerical simulations — system sizes far beyond
what was accessible in 1995 — and shows that the transition is in fact
**discontinuous (first order)**: there is phase coexistence, a latent heat
analogue, and the transition does not belong to any known equilibrium
universality class.
This is a clean pedagogical example of how finite-size effects can mask the
true nature of a phase transition: smaller systems looked like second order,
larger systems revealed first order. The debate occupied the field for nearly
a decade.

---

### Vicsek, T., & Zafeiris, A. (2012)
*Collective motion.*
Physics Reports, 517(3–4), 71–140.

A comprehensive review covering models, experiments, and theory.
Experiments range from starling murmurations and fish schools to bacterial
colonies and migrating cells. The theoretical sections connect the
agent-based description to continuum field theories and discuss open
questions around universality, fluctuations, and the role of inertia.
The best single entry point for a student who wants to go beyond the notebook
and see where the field stands.

---

### Summary table

| Paper | Key contribution |
|-------|-----------------|
| [Reynolds (1987), *SIGGRAPH* ](https://dl.acm.org/doi/10.1145/37402.37406) | three-rule algorithm; computer graphics origin |
| [Vicsek et al. (1995), *PRL 75* ](https://arxiv.org/pdf/cond-mat/0611743)| minimal physics model; order parameter φ; Ising analogy |
|[ Toner & Tu (1995, 1998), *PRL / PRE* ](https://arxiv.org/abs/adap-org/9506001) | continuum hydrodynamics; long-range order in 2-D out-of-equilibrium systems |
| [Chaté et al. (2008), *PRE 77*](https://arxiv.org/abs/0712.2062)| large-scale numerics; transition is **first order**, not second |
| [Vicsek & Zafeiris (2012), *Phys. Rep.* ](https://arxiv.org/abs/1010.5017)| comprehensive review: models, experiments, theory |